# The second report

This notebook derives `reports/faithfulness_report_2.html` from the round two
benchmark outputs of notebook 09, the same way notebook 07 derives the first
report: every number in the prose is computed from the saved tables, nothing
is typed by hand. The first report and its literature review stay untouched;
this document presents the second experiment and its four part conclusion.

## Configuration

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIRECTORY = PROJECT_ROOT / "artifacts"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
REPORT_PATH = REPORTS_DIRECTORY / "faithfulness_report_2.html"

DATASET_DIRECTORIES = {
    "imdb": ARTIFACTS_DIRECTORY / "imdb_4000",
    "yelp": ARTIFACTS_DIRECTORY / "yelp_4000",
    "agnews": ARTIFACTS_DIRECTORY / "agnews_4000",
}
DATASET_TASKS = {
    "imdb": "movie review sentiment",
    "yelp": "business review sentiment",
    "agnews": "news topic, business vs sci/tech",
}

OBSERVATION_COUNT = 300
EVALUATION_BUDGETS = [50, 200, 500]
PERCENTILES = [20, 40, 60, 80]
CHUNK_SIZES = [1, 4, 8]
RANDOM_SEED = 42

SHAP_COLOR = "#2a78d6"
LIME_COLOR = "#eb6834"
OCCLUSION_COLOR = "#1baf7a"
RANDOM_COLOR = "#898781"
GRID_COLOR = "#e1e0d9"
AXIS_COLOR = "#c3c2b7"
INK_COLOR = "#0b0b0b"
SECONDARY_INK_COLOR = "#52514e"
SURFACE_COLOR = "#fcfcfb"
METHOD_COLORS = {"shap": SHAP_COLOR, "lime": LIME_COLOR}

print(f"Report will be written to {REPORT_PATH}")

Report will be written to /home/user/projects/shapley/reports/faithfulness_report_2.html


## Load the round two outputs

In [2]:
import numpy as np
import pandas as pd

tidy = pd.read_parquet(ARTIFACTS_DIRECTORY / "faithfulness_results_v2.parquet")
timing = pd.read_parquet(ARTIFACTS_DIRECTORY / "timing_results_v2.parquet").reset_index()
method_comparison = pd.read_parquet(ARTIFACTS_DIRECTORY / "method_comparison_v2.parquet")
combination_comparison = pd.read_parquet(
    ARTIFACTS_DIRECTORY / "combination_comparison_v2.parquet"
)
metadata_by_dataset = {
    name: json.loads((directory / "metadata.json").read_text())
    for name, directory in DATASET_DIRECTORIES.items()
}

from textfaith.experiment import average_shifts_over_thresholds

average_shifts = average_shifts_over_thresholds(tidy, value_column="relative_shift")

seconds_per_review = (
    timing[timing["method"].isin(["shap", "lime"])]
    .pivot_table(
        index=["dataset", "method"], columns="budget", values="seconds_per_review"
    )
    .reindex(columns=["50", "200", "500"])
)

print(f"{len(tidy)} tidy rows, {len(method_comparison)} comparison rows")
print(seconds_per_review.round(2).to_string())

/home/user/projects/shapley/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


518400 tidy rows, 648 comparison rows
budget            50    200    500
dataset method                    
agnews  lime    0.85   2.79   6.66
        shap    0.45   1.50   3.72
imdb    lime    3.44  11.96  28.58
        shap    1.49   5.21  11.53
yelp    lime    2.54   8.67  20.83
        shap    1.10   3.68   8.02


## The points figure

SHAP and LIME at the 500 budget, mean relative shift per threshold, points
deliberately unconnected: the thresholds are separate experimental
conditions, not samples of a curve.

In [3]:
import base64
import io

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update(
    {
        "figure.facecolor": SURFACE_COLOR,
        "axes.facecolor": SURFACE_COLOR,
        "savefig.facecolor": SURFACE_COLOR,
        "axes.edgecolor": AXIS_COLOR,
        "axes.labelcolor": SECONDARY_INK_COLOR,
        "text.color": INK_COLOR,
        "xtick.color": SECONDARY_INK_COLOR,
        "ytick.color": SECONDARY_INK_COLOR,
        "grid.color": GRID_COLOR,
        "font.size": 11,
        "savefig.dpi": 150,
        "savefig.bbox": "tight",
    }
)

report_images = {}


def figure_to_data_uri(figure) -> str:
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png")
    plt.close(figure)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"


POINTS_CHUNK_SIZE = 4
POINTS_COMBINATION = "absolute_sum"

figure, axes = plt.subplots(2, 3, figsize=(12.5, 7.0), sharey="row", sharex=True)
for column_index, dataset_name in enumerate(DATASET_DIRECTORIES):
    for row_index, direction in enumerate(["sufficiency", "comprehensiveness"]):
        axis = axes[row_index][column_index]
        for method, offset in [("shap", -1.5), ("lime", 1.5)]:
            subset = tidy[
                (tidy["dataset"] == dataset_name)
                & (tidy["source"] == f"{method}_500")
                & (tidy["chunk_size"] == POINTS_CHUNK_SIZE)
                & (tidy["combination"] == POINTS_COMBINATION)
                & (tidy["direction"] == direction)
            ]
            means = subset.groupby("percentile")["relative_shift"].mean() * 100
            axis.plot(
                means.index + offset, means.to_numpy(),
                linestyle="", marker="o", markersize=9,
                color=METHOD_COLORS[method],
                label=method.upper() if row_index == 0 and column_index == 0 else None,
            )
        axis.set_xticks(PERCENTILES)
        axis.grid(alpha=0.6, linewidth=0.7)
        for spine_name in ["top", "right"]:
            axis.spines[spine_name].set_visible(False)
        if row_index == 0:
            axis.set_title(dataset_name, fontsize=12)
        else:
            axis.set_xlabel("percent of text cut")

axes[0][0].set_ylabel("sufficiency, % of confidence")
axes[1][0].set_ylabel("comprehensiveness, % of confidence")
axes[0][0].legend(fontsize=9, frameon=False)
figure.suptitle(
    f"Mean relative shift per threshold, budget 500, chunk size "
    f"{POINTS_CHUNK_SIZE}, {POINTS_COMBINATION}",
    y=1.0, fontsize=13,
)
figure.tight_layout()
report_images["points"] = figure_to_data_uri(figure)
print("points figure built")

points figure built


## Headline statistics for the prose

In [4]:
def family_summary(comparison_name):
    subset = method_comparison.loc[comparison_name]
    survivors = subset[subset["holm_significant"]]
    return {
        "total": int(len(subset)),
        "significant": int(len(survivors)),
        "first": int((survivors["first_advantage"] > 0).sum()),
        "second": int((survivors["first_advantage"] < 0).sum()),
        "mean_advantage_percent": float(subset["first_advantage"].mean()) * 100,
    }


random_families = {
    first: family_summary(f"{first} vs random")
    for first in ["shap_500", "lime_500", "occlusion"]
}
head_to_head = family_summary("shap_500 vs lime_500")
occlusion_families = {
    second: family_summary(f"occlusion vs {second}")
    for second in ["shap_500", "lime_500"]
}
budget_families = {
    name: family_summary(name)
    for name in [
        "shap_500 vs shap_200", "shap_200 vs shap_50", "shap_500 vs shap_50",
        "lime_500 vs lime_200", "lime_200 vs lime_50", "lime_500 vs lime_50",
    ]
}

for method, family in random_families.items():
    assert family["second"] == 0, (
        f"{method} lost a condition to random, the 4.1 prose must change"
    )
for name in [
    "shap_500 vs shap_200", "shap_200 vs shap_50",
    "lime_500 vs lime_200", "lime_200 vs lime_50", "lime_500 vs lime_50",
]:
    assert budget_families[name]["second"] == 0, (
        f"the smaller budget won in {name}, the 4.3 prose must change"
    )
shap_end_to_end_reversals = budget_families["shap_500 vs shap_50"]["second"]

absolute_sum_rows = combination_comparison[
    combination_comparison.index.get_level_values("comparison").str.startswith(
        "absolute_sum vs"
    )
]
assert (absolute_sum_rows["significant_for_second"] == 0).all(), (
    "absolute_sum lost a combination condition, the 4.4 prose must change"
)
absolute_sum_wins = int(absolute_sum_rows["significant_for_first"].sum())
absolute_sum_conditions = int(absolute_sum_rows["conditions"].sum())

MAIN_SOURCES = ["shap_500", "lime_500", "occlusion", "random"]
relative_table = (
    tidy[
        tidy["source"].isin(MAIN_SOURCES)
        & (tidy["combination"] == "absolute_sum")
    ]
    .pivot_table(
        values="relative_shift",
        index=["dataset", "chunk_size"],
        columns=["direction", "source"],
        aggfunc="mean",
    )
    .reindex(
        columns=pd.MultiIndex.from_product(
            [["sufficiency", "comprehensiveness"], MAIN_SOURCES]
        )
    )
    * 100
)

budget_table = (
    average_shifts[average_shifts["method"].isin(["shap", "lime"])]
    .pivot_table(
        values="average_shift",
        index=["dataset", "method"],
        columns=["direction", "budget"],
        aggfunc="mean",
    )
    .reindex(
        columns=pd.MultiIndex.from_product(
            [["sufficiency", "comprehensiveness"], ["50", "200", "500"]]
        )
    )
    * 100
)

combination_means = (
    average_shifts[
        average_shifts["source"].isin(["shap_500", "lime_500"])
        & average_shifts["chunk_size"].isin([4, 8])
    ]
    .groupby(["source", "direction", "combination"])["average_shift"]
    .mean()
    .unstack("combination")
    * 100
)

comparison_summary_rows = []
for comparison_name in method_comparison.index.get_level_values("comparison").unique():
    summary = family_summary(comparison_name)
    comparison_summary_rows.append({"comparison": comparison_name} | summary)
comparison_summary = pd.DataFrame(comparison_summary_rows).set_index("comparison")

shap_seconds = seconds_per_review.xs("shap", level="method")
lime_seconds = seconds_per_review.xs("lime", level="method")
median_words = {
    name: int(
        pd.read_parquet(directory / "test.parquet")["text"].str.split().str.len().median()
    )
    for name, directory in DATASET_DIRECTORIES.items()
}
accuracy_range = (
    min(metadata["test_accuracy"] for metadata in metadata_by_dataset.values()),
    max(metadata["test_accuracy"] for metadata in metadata_by_dataset.values()),
)

print(comparison_summary.round(2).to_string())
print()
print(f"median test words: {median_words}")

                       total  significant  first  second  mean_advantage_percent
comparison                                                                      
shap_500 vs random        54           54     54       0                   10.15
lime_500 vs random        54           54     54       0                    8.26
occlusion vs random       54           52     52       0                    6.49
shap_500 vs lime_500      54           31     25       6                    1.89
occlusion vs shap_500     54           29      2      27                   -3.66
occlusion vs lime_500     54           24      0      24                   -1.77
shap_500 vs shap_200      54            8      8       0                    0.69
shap_200 vs shap_50       54           14     14       0                    1.80
shap_500 vs shap_50       54           19     15       4                    2.48
lime_500 vs lime_200      54           12     12       0                    1.34
lime_200 vs lime_50       54

## Assemble and write the report

In [5]:
REPORT_CSS = '''
:root { color-scheme: light; }
body {
  background: #f9f9f7; color: #0b0b0b; margin: 0;
  font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
  line-height: 1.55;
}
main { max-width: 60rem; margin: 0 auto; padding: 2.5rem 1.5rem 4rem; }
h1 { font-size: 1.9rem; line-height: 1.2; margin: 0 0 0.4rem; }
h2 { font-size: 1.35rem; margin: 2.6rem 0 0.8rem; }
h3 { font-size: 1.05rem; margin: 1.8rem 0 0.5rem; }
p, li { max-width: 46rem; }
.byline { color: #52514e; margin-bottom: 2rem; }
.abstract {
  background: #fcfcfb; border: 1px solid rgba(11,11,11,0.10);
  border-radius: 8px; padding: 1.1rem 1.4rem; margin: 1.5rem 0;
}
.abstract p { margin: 0.4rem 0; }
figure { margin: 1.6rem 0; }
figure img { max-width: 100%; height: auto; background: #fcfcfb;
  border: 1px solid rgba(11,11,11,0.10); border-radius: 8px; }
figcaption { color: #52514e; font-size: 0.9rem; margin-top: 0.5rem; max-width: 46rem; }
.table-wrap { overflow-x: auto; margin: 1.2rem 0; }
table { border-collapse: collapse; font-size: 0.85rem; background: #fcfcfb; }
th, td { padding: 0.35rem 0.7rem; border: 1px solid #e1e0d9; text-align: right;
  font-variant-numeric: tabular-nums; }
th { background: #f0efec; color: #0b0b0b; text-align: right; }
td:first-child, th:first-child { text-align: left; }
code { background: #f0efec; padding: 0.1rem 0.3rem; border-radius: 4px;
  font-size: 0.85em; }
a { color: #256abf; }
.formula { font-family: Georgia, "Times New Roman", serif;
  text-align: center; margin: 1.1rem auto; font-size: 1.02rem; }
.footer { color: #898781; font-size: 0.85rem; margin-top: 3rem;
  border-top: 1px solid #e1e0d9; padding-top: 1rem; }
'''


def frame_to_html(frame, caption):
    table_html = frame.to_html(
        float_format=lambda value: f"{value:.2f}", border=0
    )
    return (
        f'<div class="table-wrap">{table_html}</div>'
        f'<figcaption>{caption}</figcaption>'
    )


setup_rows = []
for dataset_name, metadata in metadata_by_dataset.items():
    setup_rows.append(
        {
            "dataset": dataset_name,
            "task": DATASET_TASKS[dataset_name],
            "train / val / test": (
                f"{metadata['train_size']} / {metadata['validation_size']} / "
                f"{metadata['test_size']}"
            ),
            "validation accuracy": metadata["validation_accuracy"],
            "test accuracy": metadata["test_accuracy"],
            "median test words": median_words[dataset_name],
            "explained": OBSERVATION_COUNT,
        }
    )
setup_table = pd.DataFrame(setup_rows).set_index("dataset")

sections = []

sections.append(f'''
<h1>Quantifying the Faithfulness of Text Classification Explanations</h1>
<p class="byline">Second experiment: four attribution methods, three
evaluation budgets, {OBSERVATION_COUNT} test documents per dataset &middot;
companion to the first report, which carries the literature review &middot;
seed {RANDOM_SEED}</p>

<div class="abstract">
<h3 style="margin-top:0">Abstract</h3>
<p>This round re-runs the deletion faithfulness benchmark on stronger
foundations: 4000 document checkpoints with a proper 70/15/15 train,
validation, test split (test accuracy {accuracy_range[0]:.2f} to
{accuracy_range[1]:.2f}), explanation evaluated exclusively on the test
split, {OBSERVATION_COUNT} stratified documents of its 600 per dataset,
three times round one. The roster grows to four methods, SHAP, LIME, leave
one word out occlusion, and an explicit gaussian random baseline that
replaces the shuffled controls, and the budget axis grows to three points,
50, 200 and 500 model evaluations for SHAP and LIME. All results are
reported as a percentage of the model's original predicted class
confidence.</p>
<p>Every real method beats the random baseline decisively, SHAP in
{random_families["shap_500"]["first"]} of 54 Holm corrected conditions,
LIME in {random_families["lime_500"]["first"]}, occlusion in
{random_families["occlusion"]["first"]}, and none ever loses to it, so the
metric separates knowledge from noise. Head to head at 500 evaluations,
{head_to_head["first"]} conditions favor SHAP and {head_to_head["second"]}
favor LIME of {head_to_head["significant"]} significant; SHAP costs
{shap_seconds["500"].mean():.1f} seconds per explained document on average
against LIME's {lime_seconds["500"].mean():.1f}. Across every method, direction
and chunked condition, <code>absolute_sum</code> never loses a Holm corrected
comparison against the other two combinations; the budget ladders are
resolved in the conclusion.</p>
</div>
''')

sections.append(f'''
<h2>1. Experimental setup</h2>
<p>Three binary text classification datasets, one shared pipeline: frozen
<code>e5-small-v2</code> embeddings, a single hidden layer MLP head, 4000
stratified documents per dataset split 70/15/15. The head's early stopping
uses an internal slice of the training set, so the validation split stays
clean and the test split is used for exactly one thing, explanation
evaluation. All explained documents are drawn from the test split,
{OBSERVATION_COUNT} of 600, stratified on the label.</p>
{frame_to_html(setup_table, "Table 1. Round two datasets and models. Validation and test accuracies agree within sampling noise, so the heads are not overfit to their stopping criterion.")}
<p>Four attribution methods produce one signed value per whitespace word:
<b>SHAP</b> (partition explainer, budget capped), <b>LIME</b> (ridge
surrogate over word dropouts, budget exact), <b>occlusion</b> (leave one
word out, cost fixed at one prediction per word) and <b>random</b>, an
independent standard gaussian draw per word that costs no predictions and
replaces round one's per method shuffled controls. SHAP and LIME run at 50,
200 and 500 evaluations; cross method comparisons use the 500 budget.
Faithfulness is measured exactly as in round one: cut chunks of 1, 4 or 8
words ranked by the attribution under three combinations, at thresholds of
20 to 80 percent of the document, in both directions, and average the
threshold results into one score per document, expressed relative to the
model's original confidence.</p>
''')

sections.append(f'''
<h2>2. Definitions</h2>
<p>Write <i>f</i>(<i>s</i>) for the model's probability of the predicted
class on text <i>s</i>, fixed from the full text. With cut thresholds
<i>t</i><sub>1</sub> &lt; &hellip; &lt; <i>t</i><sub><i>n</i></sub> (20, 40,
60, 80), <i>s</i>&#8725;<i>t</i><sup>&minus;</sup> the text with chunks below
the <i>t</i>-th percentile of its own chunk importances removed, and
<i>s</i>&#8725;<i>t</i><sup>+</sup> its most important mirror:</p>
<p class="formula">
sufficiency(<i>s</i>) = (1/<i>n</i>) &Sigma;<sub><i>i</i></sub>
&#124;<i>f</i>(<i>s</i>) &minus; <i>f</i>(<i>s</i>&#8725;<i>t</i><sub><i>i</i></sub><sup>&minus;</sup>)&#124; / <i>f</i>(<i>s</i>)
</p>
<p class="formula">
comprehensiveness(<i>s</i>) = (1/<i>n</i>) &Sigma;<sub><i>i</i></sub>
&#124;<i>f</i>(<i>s</i>) &minus; <i>f</i>(<i>s</i>&#8725;<i>t</i><sub><i>i</i></sub><sup>+</sup>)&#124; / <i>f</i>(<i>s</i>)
</p>
<p>Both are shares of the model's original confidence, in [0, 1], reported
in percent. Sufficiency is better when lower, comprehensiveness when higher.
The first report derives these step by step with a worked example; this one
uses them directly.</p>
''')

sections.append(f'''
<h2>3. Results</h2>

<h3>3.1 Seconds per explained document</h3>
{frame_to_html(seconds_per_review.round(2), "Table 2. Wall clock seconds per explained document for SHAP and LIME at each budget, 8 core CPU. Occlusion and random are omitted: occlusion's cost is structural (one prediction per word) and random's is zero.")}

<h3>3.2 The landscape at the comparison budget</h3>
<figure>
<img src="{report_images["points"]}" alt="Mean relative shift per threshold for SHAP and LIME at budget 500, shown as unconnected points">
<figcaption>Figure 1. Mean relative shift per threshold at budget 500, chunk
size 4, absolute_sum. Points are deliberately unconnected: thresholds are
separate conditions, not a sampled curve. Top row lower is better, bottom row
higher is better.</figcaption>
</figure>
{frame_to_html(relative_table.round(1), "Table 3. Mean shift as percent of the model's confidence, absolute_sum, averaged over thresholds and observations: the four methods at their comparison budget, random as the floor.")}

<h3>3.3 Pairwise comparisons</h3>
{frame_to_html(comparison_summary, "Table 4. Holm corrected outcome of all twelve pairwise comparisons, 54 conditions each. Mean advantage is in percentage points of confidence, positive favoring the first named source.")}

<h3>3.4 The budget ladder</h3>
{frame_to_html(budget_table.round(1), "Table 5. Threshold averaged relative scores per budget, pooled over chunk sizes and combinations, in percent of confidence.")}

<h3>3.5 Chunk combinations</h3>
{frame_to_html(combination_means.round(2), "Table 6. Mean relative score per combination at the 500 budget, chunk sizes 4 and 8 pooled (at chunk size 1 the combinations coincide). Sufficiency better lower, comprehensiveness better higher.")}
{frame_to_html(combination_comparison.round(2), "Table 7. Pairwise combination comparisons per method and direction: Wilcoxon per condition (3 datasets x 2 chunked sizes), Holm corrected per family of six, positive advantage favoring the first named combination.")}
''')

sections.append(f'''
<h2>4. Conclusion</h2>

<h3>4.1 Does the metric make sense? The random attribution test</h3>
<p>A faithfulness metric is meaningful only if knowing something beats
knowing nothing. It does, without exception: against the gaussian random
baseline, SHAP wins {random_families["shap_500"]["first"]},
LIME {random_families["lime_500"]["first"]} and occlusion
{random_families["occlusion"]["first"]} of 54 Holm corrected conditions
each, and no method loses a single condition to random in either direction.
The margins are large as well as significant, mean advantages of
{random_families["shap_500"]["mean_advantage_percent"]:.1f},
{random_families["lime_500"]["mean_advantage_percent"]:.1f} and
{random_families["occlusion"]["mean_advantage_percent"]:.1f} percentage
points of confidence respectively. The metric separates informed rankings
from noise; what remains is separating informed rankings from each other.</p>

<h3>4.2 SHAP or LIME?</h3>
<p>At the shared 500 budget the verdict is no longer close: of 54
conditions, {head_to_head["significant"]} separate the methods after Holm
correction, {head_to_head["first"]} favoring SHAP and
{head_to_head["second"]} favoring LIME, with a mean advantage of
{head_to_head["mean_advantage_percent"]:+.1f} percentage points of
confidence toward SHAP. LIME's wins cluster where round one already found
them, word level sufficiency on the news topic task, whose signal
concentrates in a few entity terms; SHAP takes comprehensiveness nearly
everywhere and sufficiency on both sentiment tasks. On cost the gap
compounds: SHAP explains a document in {shap_seconds["500"].mean():.1f}
seconds on average against LIME's {lime_seconds["500"].mean():.1f} at the
same budget, a factor of
{lime_seconds["500"].mean() / shap_seconds["500"].mean():.1f}, because
SHAP's masker collapses removed spans while LIME's perturbed texts stay near
full length. More faithful on most conditions and less than half the price:
for this pipeline the head to head goes to SHAP.</p>

<h3>4.3 Which evaluation budget is enough?</h3>
<p>These are rather short texts, medians of {median_words["imdb"]},
{median_words["yelp"]} and {median_words["agnews"]} words, so the three
budgets bracket the word counts: 50 evaluations sits below every median, 500
above all of them. The answer differs sharply by method. LIME needs budget:
its full arm beats the 50 arm in
{budget_families["lime_500 vs lime_50"]["first"]} of 54 conditions, with
both intermediate rungs significant
({budget_families["lime_500 vs lime_200"]["first"]} and
{budget_families["lime_200 vs lime_50"]["first"]} conditions), consistent
with a ridge surrogate that is underdetermined until its sample count clears
the word count. SHAP is nearly budget saturated on these texts: 500 beats
200 in only {budget_families["shap_500 vs shap_200"]["first"]} of 54
conditions, the gains concentrating in comprehensiveness while sufficiency
is flat, and the end to end comparison even reverses in
{shap_end_to_end_reversals} word level sufficiency conditions where the
coarse 50 evaluation partition is marginally better. The practical
recommendation for texts of this length: SHAP at 200 evaluations is already
a sound operating point and 50 is usable, while LIME should not be run below
200 and prefers 500; at the smallest budget SHAP is the only sensible
choice of the two.</p>

<h3>4.4 Which chunk combination?</h3>
<p>Judged per method on the chunked sizes, where the combinations actually
differ (Tables 6 and 7), the answer is unusually clean:
<code>absolute_sum</code> is the best mean for both methods in both
directions, and across all {absolute_sum_conditions} Holm corrected
pairwise conditions involving it, it wins {absolute_sum_wins} and loses
none. The margins are largest in comprehensiveness, over one and a half
percentage points of confidence against either alternative, exactly where a
span's ability to cancel opposing words matters most. The other two
combinations are statistically indistinguishable from each other. This is
the outcome theory predicts for additive attributions, the signed sum of a
chunk is its expected output shift, and it holds empirically for LIME's
non additive weights as well: whichever of SHAP or LIME is used,
<code>absolute_sum</code> is the combination to choose, and the ranking of
methods reported above is unchanged under either alternative.</p>
''')

sections.append(f'''
<div class="footer">
<p>Reproduction: notebooks <code>08_datasets_v2</code> (checkpoints),
<code>09_benchmark_v2</code> (attributions, sweeps, statistics),
<code>10_report_2</code> (this document). Tidy results in
<code>faithfulness_results_v2.parquet</code>, timing in
<code>timing_results_v2.parquet</code>, comparisons in
<code>method_comparison_v2.parquet</code> and
<code>combination_comparison_v2.parquet</code>. Seed {RANDOM_SEED}
throughout. The companion report carries the related work, references and
worked examples.</p>
</div>
''')

report_body = "\n".join(sections)
report_document = (
    "<!doctype html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n"
    "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n"
    "<title>Quantifying the Faithfulness of Text Classification Explanations, Round 2</title>\n"
    f"<style>{REPORT_CSS}</style>\n</head>\n<body>\n<main>\n"
    f"{report_body}\n</main>\n</body>\n</html>\n"
)

REPORTS_DIRECTORY.mkdir(exist_ok=True)
REPORT_PATH.write_text(report_document)
print(f"Wrote {REPORT_PATH} ({len(report_document) / 1024:.0f} KiB, "
      f"{len(report_images)} embedded figures)")

Wrote /home/user/projects/shapley/reports/faithfulness_report_2.html (174 KiB, 1 embedded figures)
